In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("merge_keys", "", "Merge Keys")
dbutils.widgets.text("load_type", "", "Load Type")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

print("[STEP 1] Parameters received:")
print(f"  table_name        = {table_name}")
print(f"  landing_timestamp = {landing_timestamp}")
print(f"  load_type         = {load_type}")
print(f"  merge_keys        = {merge_keys}")

In [0]:
print(f"[STEP 2] Reading bronze table caresync_databricks.bronze.{table_name} for landing_timestamp = '{landing_timestamp}'")
bronze_df = spark.read.table(f"caresync_databricks.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")
print(f"[STEP 2] Bronze rows read: {bronze_df.count()}")
display(bronze_df)

In [0]:
from pyspark.sql.functions import current_timestamp
print("[STEP 3] Adding insert_timestamp and update_timestamp columns")
updated_df = bronze_df.withColumn("insert_timestamp",current_timestamp()).withColumn("update_timestamp",current_timestamp())
print(f"[STEP 3] Columns: {updated_df.columns}")

In [0]:
from delta.tables import DeltaTable

target_table = f"caresync_databricks.silver.{table_name}"
load_type_norm = load_type.strip().upper()
print(f"[STEP 4] Target table: {target_table} | load_type: {load_type_norm}")

if load_type_norm == "FULL":
    # Full load: overwrite the target table
    print("[STEP 4] FULL load: overwriting target table...")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"[STEP 4] FULL load complete. Target row count: {spark.table(target_table).count()}")

elif load_type_norm == "APPEND":
    # Append: add the new batch to the target table
    print("[STEP 4] APPEND load: appending rows to target table...")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[STEP 4] APPEND complete. Target row count: {spark.table(target_table).count()}")

elif load_type_norm == "MERGE":
    # Merge (upsert): update existing records, insert new ones based on merge keys
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    print(f"[STEP 4] MERGE load: merge keys = {keys}")
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type = 'MERGE'")

    missing = [k for k in keys if k not in updated_df.columns]
    if missing:
        raise ValueError(f"Merge keys not found in source data: {missing}")

    # Keep one row per key to avoid multiple-source-match errors in MERGE
    source_df = updated_df.dropDuplicates(keys)
    print(f"[STEP 4] Source rows after de-duplication on merge keys: {source_df.count()}")

    if not spark.catalog.tableExists(target_table):
        # First run: create the target table
        print("[STEP 4] Target table does not exist. Creating it from source data...")
        source_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[STEP 4] Target table created. Row count: {spark.table(target_table).count()}")
    else:
        merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        print(f"[STEP 4] Target table exists. Merge condition: {merge_condition}")
        # On update keep the original insert_timestamp
        update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsert(values=insert_set)
            .execute())

        # Show inserted/updated counts from the latest Delta commit
        metrics = spark.sql(f"DESCRIBE HISTORY {target_table} LIMIT 1").collect()[0]["operationMetrics"]
        print(f"[STEP 4] MERGE complete. Rows inserted: {metrics.get('numTargetRowsInserted')}, "
              f"rows updated: {metrics.get('numTargetRowsUpdated')}")
        print(f"[STEP 4] Target row count: {spark.table(target_table).count()}")

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

In [0]:
if "updated_df" not in globals():
    updated_df = spark.read.table(f"caresync_databricks.bronze.{dbutils.widgets.get('table_name')}").filter(f"landing_timestamp = '{dbutils.widgets.get('landing_timestamp')}'")
record_count = updated_df.count()
print(f"[STEP 5] Exiting notebook. Records processed: {record_count}")
dbutils.notebook.exit(str(record_count))